# Build Your Own Benchmark (BYOB): MCQ pipeline

## Prerequisite: target corpus (`.txt` files)

You need a **folder of plain-text files** that BYOB uses as the domain corpus (wired through `input_dir` and `target_source_mapping` in **`default.yaml`**).

- **Use your own data:** put one or more `.txt` files in a directory (for example under `./assets/<your_folder>/`) and point the YAML at that path.
- **Or build a corpus from Wikipedia:** run [`download_wikipedia_data.ipynb`](download_wikipedia_data.ipynb) in the same directory. It can download articles from a Wikipedia **category** (and related options) and write a folder of `.txt` files—use that folder as your corpus.

Do this **before** preparing seed data so the paths in `default.yaml` match an existing folder.

---

Run this notebook **top to bottom**:

1. **Paths** — locate `default.yaml` and ensure the Nemotron `nemotron` package is importable (the next cell adds the repo `src` tree to `sys.path` if needed).
2. **API keys** — set `NVIDIA_API_KEY` for NIM and optionally `HF_TOKEN` for Hugging Face model downloads (cells right after paths).
3. **Prepare seed data** — builds `seed.parquet` from few-shot examples (e.g. MMLU) plus your corpus under `input_dir` (see `default.yaml`).
4. **Run the MCQ pipeline** — section **3** runs one markdown + code cell per stage (3.0 setup, then 3.1–3.9).

Configuration is **`default.yaml`** in this folder (paths are relative to this directory).

## Before you start

- **Working directory:** Jupyter’s cwd should be **this notebook’s folder** so `./assets/...` and `./default.yaml` resolve correctly.
- **Corpus:** See **Prerequisite** above—either your own `.txt` folder or outputs from [`download_wikipedia_data.ipynb`](download_wikipedia_data.ipynb). Align `default.yaml` with that folder (the sample uses `./assets/wikipedia_finance_india/`).
- **Models:** `default.yaml` uses **NVIDIA NIM**-style models (`provider: nvidia`). Use the **Set NVIDIA_API_KEY** and optional **Set HF_TOKEN** cells after paths, or export `NVIDIA_API_KEY` / `HF_TOKEN` before starting Jupyter. Set any other env vars your NeMo Data Designer setup needs.
- **Cost & time:** The pipeline calls LLMs and embeddings on many rows — plan quota and runtime.
- **Install:** `pip install -e .` from your Nemotron checkout if you use an editable install; otherwise the first code cell prepends the Nemotron `src` directory to `sys.path` for this session.


## 1. Paths and configuration file

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
CONFIG_YAML = NOTEBOOK_DIR / "default.yaml"

if not CONFIG_YAML.is_file():
    raise RuntimeError(
        "Could not find default.yaml. Open or restart Jupyter with cwd set to "
        "the folder containing this notebook and default.yaml."
    )

NEMOTRON_REPO_ROOT = NOTEBOOK_DIR.parent.parent
NEMOTRON_SRC = NEMOTRON_REPO_ROOT / "src"
if str(NEMOTRON_SRC) not in sys.path:
    sys.path.insert(0, str(NEMOTRON_SRC))

_sep = os.pathsep
_pp = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = str(NEMOTRON_SRC) + (_sep + _pp if _pp else "")

print("Notebook dir:", NOTEBOOK_DIR)
print("Config:", CONFIG_YAML)
print("Nemotron src on PYTHONPATH:", NEMOTRON_SRC)

### Set `NVIDIA_API_KEY`

NVIDIA NIM–style endpoints (see `default.yaml`) use **`NVIDIA_API_KEY`**. The next cell assigns it for this Jupyter session if needed, or verifies it is already set in your environment.


In [ ]:
import os

NVIDIA_API_KEY = None

if NVIDIA_API_KEY:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

if not os.environ.get("NVIDIA_API_KEY"):
    raise RuntimeError(
        "NVIDIA_API_KEY is not set. Assign NVIDIA_API_KEY in this cell or run: "
        "export NVIDIA_API_KEY=... before starting Jupyter."
    )

### Set `HF_TOKEN` (Hugging Face)

Embedding and tokenizer stages download models from the Hugging Face Hub. **Public models** work without a token; set **`HF_TOKEN`** if you use **gated** models, need higher rate limits, or hit authentication errors during downloads. The next cell assigns it for this session (or leave `None` to use `HF_TOKEN` / `HUGGING_FACE_HUB_TOKEN` already set in your shell).

In [ ]:
import os

# Hugging Face token (e.g. "hf_..."); None = do not override — use existing shell env
HF_TOKEN = None

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    # huggingface_hub also reads this alias
    os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", HF_TOKEN)


In [ ]:
!export RAY_EXPERIMENTAL_NOSET_CUDA_VISIBLE_DEVICES=0

## 2. Prepare seed data

Loads `ByobConfig`, builds the dataset with `make_from_config`, and writes **`seed.parquet`** to `<output_dir>/<expt_name>/`. **Run this before the pipeline** — downstream stages depend on that seed.

In [ ]:
from nemotron.steps.byob.config import ByobConfig
from nemotron.steps.byob.dataset import make_from_config

config = ByobConfig.from_yaml(str(CONFIG_YAML))
dataset = make_from_config(config)
seed_df = dataset.sample_and_dump()
seed_df

## 3. MCQ generation pipeline

Run **section 3.0** first (imports and paths), then **3.1 through 3.9** in order. Each stage reads the previous Parquet (`last_path`) and writes the next file under `<output_dir>/<expt_name>/stage_cache/`.

Optional stages **3.4** and **3.5** run only when `do_distractor_expansion` and `do_coverage_check` are enabled in `default.yaml`.

**Requires:** section 2 completed (`seed.parquet`). Stages call LLMs/embeddings — plan time and cost.


### 3.0 Pipeline setup — imports and output paths

Imports NeMo Data Designer helpers and MCQ utilities, configures logging, and defines path variables (`paths`, `output_path_raw`, `output_path_final`). Run this once before the stages below; it uses `config` from section 2.


In [ ]:
import logging
import os

import pandas as pd

from nemotron.steps.byob.data_designer_utils import batched_run
from nemotron.steps.byob.mcq.deduplication import TextSemanticDeduplicationMCQ
from nemotron.steps.byob.mcq.semantic_outlier import TextSemanticOutlierDetectionMCQ
from nemotron.steps.byob.mcq.stages import (
    check_distractor_validity,
    expand_distractors,
    filter_questions,
    generate_questions,
    judge_questions,
)
from nemotron.steps.byob.mcq.text_coverage import TextCoverageMCQ
from nemotron.steps.byob.mcq.utils import (
    postprocess_distractor_expansion,
    postprocess_filtered_questions,
    postprocess_generated_questions,
    postprocess_judged_questions,
    postprocess_distractor_validity,
    prepare_distractor_expansion_seed_dataset,
    prepare_distractor_validity_seed_dataset,
    prepare_filtering_seed_dataset,
    prepare_generation_seed_dataset,
    prepare_judgement_seed_dataset,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

output_base = os.path.join(config.output_dir, config.expt_name)
stage_cache = os.path.join(output_base, "stage_cache")
os.makedirs(stage_cache, exist_ok=True)

paths = {
    "generation": os.path.join(stage_cache, "generated_questions.parquet"),
    "judgement": os.path.join(stage_cache, "judged_questions.parquet"),
    "semantic_dedup": os.path.join(stage_cache, "semantic_deduplicated_questions.parquet"),
    "distractor_expansion": os.path.join(stage_cache, "expanded_distractors.parquet"),
    "coverage": os.path.join(stage_cache, "coverage_check.parquet"),
    "distractor_validity": os.path.join(stage_cache, "valid_distractors.parquet"),
    "semantic_outlier": os.path.join(stage_cache, "semantic_outlier_detection.parquet"),
    "filtering": os.path.join(stage_cache, "filtered_questions.parquet"),
}
output_path_raw = os.path.join(output_base, "benchmark_raw.parquet")
output_path_final = os.path.join(output_base, "benchmark.parquet")

last_path = None

print("Output base:", output_base)
print("Stage cache:", stage_cache)


### 3.1 Question generation

Builds the generation seed from **`seed.parquet`**, calls NeMo Data Designer (`generate_questions`) to create MCQs tied to your corpus, postprocesses, and saves **`generated_questions.parquet`**.


In [ ]:
seed_df_generation = prepare_generation_seed_dataset(config)
dataset_out = batched_run(
    generate_questions,
    config,
    seed_df_generation,
    batch_size=config.ndd_batch_size,
)
dataset_out.dropna(inplace=True)
dataset_out = postprocess_generated_questions(dataset_out)
dataset_out.to_parquet(paths["generation"])
last_path = paths["generation"]
logger.info("Saved: %s", last_path)

### 3.2 Judgement

Runs an LLM pass (`judge_questions`) to assess each generated item, merges results with the generation output, and writes **`judged_questions.parquet`**.


In [ ]:
dataset_in = pd.read_parquet(paths["generation"])
seed_j = prepare_judgement_seed_dataset(config, dataset_in)
dataset_out = batched_run(
    judge_questions,
    config,
    seed_j,
    batch_size=config.ndd_batch_size,
)
dataset_out = postprocess_judged_questions(dataset_in, dataset_out)
dataset_out.to_parquet(paths["judgement"])
last_path = paths["judgement"]
logger.info("Saved: %s", last_path)


### 3.3 Semantic deduplication

Uses `TextSemanticDeduplicationMCQ` (embeddings) to drop near-duplicate questions. Output: **`semantic_deduplicated_questions.parquet`**.


In [ ]:
dataset_in = pd.read_parquet(last_path)
dedup = TextSemanticDeduplicationMCQ(config)
dataset_out = dedup.run(dataset_in)
dataset_out.to_parquet(paths["semantic_dedup"])
last_path = paths["semantic_dedup"]
logger.info("Saved: %s", last_path)


### 3.4 Distractor expansion (optional)

If `do_distractor_expansion: true`, calls `expand_distractors` to add or refine wrong choices. Updates `last_path` to **`expanded_distractors.parquet`**. If disabled, logs a skip and leaves `last_path` on the semantic-dedup file.


In [ ]:
if config.do_distractor_expansion:
    logger.info("Expanding distractors...")
    dataset_in = pd.read_parquet(last_path)
    seed_de = prepare_distractor_expansion_seed_dataset(config, dataset_in)
    dataset_out = batched_run(
        expand_distractors,
        config,
        seed_de,
        batch_size=config.ndd_batch_size,
    )
    dataset_out = postprocess_distractor_expansion(dataset_in, dataset_out)
    dataset_out.to_parquet(paths["distractor_expansion"])
    last_path = paths["distractor_expansion"]
    logger.info("Saved: %s", last_path)
else:
    logger.info("Skipping distractor expansion (do_distractor_expansion is false).")


### 3.5 Text coverage check (optional)

If `do_coverage_check: true`, `TextCoverageMCQ` analyzes how well questions reflect the source text. Writes **`coverage_check.parquet`** when enabled; otherwise `last_path` stays on the prior stage.


In [ ]:
if config.do_coverage_check:
    logger.info("Coverage check...")
    dataset_in = pd.read_parquet(last_path)
    cov = TextCoverageMCQ(config)
    dataset_out = cov.analyze(dataset_in)
    dataset_out.to_parquet(paths["coverage"])
    last_path = paths["coverage"]
    logger.info("Saved: %s", last_path)
else:
    logger.info("Skipping coverage check (do_coverage_check is false).")


### 3.6 Distractor validity

LLM stage (`check_distractor_validity`) checks whether incorrect options are plausible distractors. Saves **`valid_distractors.parquet`**.


In [ ]:
dataset_in = pd.read_parquet(last_path)
seed_dv = prepare_distractor_validity_seed_dataset(config, dataset_in)
dataset_out = batched_run(
    check_distractor_validity,
    config,
    seed_dv,
    batch_size=config.ndd_batch_size,
)
dataset_out = postprocess_distractor_validity(dataset_in, dataset_out)
dataset_out.to_parquet(paths["distractor_validity"])
last_path = paths["distractor_validity"]
logger.info("Saved: %s", last_path)


### 3.7 Semantic outlier detection

`TextSemanticOutlierDetectionMCQ` flags answer choices that are semantically misaligned with the question stem. Output: **`semantic_outlier_detection.parquet`**.


In [ ]:
dataset_in = pd.read_parquet(last_path)
out_det = TextSemanticOutlierDetectionMCQ(config)
dataset_out = out_det.detect(dataset_in)
dataset_out.to_parquet(paths["semantic_outlier"])
last_path = paths["semantic_outlier"]
logger.info("Saved: %s", last_path)


### 3.8 Hallucination and easiness filtering

`filter_questions` labels items that may be hallucinated or trivially easy. Output: **`filtered_questions.parquet`**.


In [ ]:
dataset_in = pd.read_parquet(last_path)
seed_f = prepare_filtering_seed_dataset(dataset_in, config)
dataset_out = batched_run(
    filter_questions,
    config,
    seed_f,
    batch_size=config.ndd_batch_size,
)
dataset_out = postprocess_filtered_questions(dataset_in, dataset_out, config)
dataset_out.to_parquet(paths["filtering"])
last_path = paths["filtering"]
logger.info("Saved: %s", last_path)


### 3.9 Final benchmark export

Writes **`benchmark_raw.parquet`**, optionally removes rows per `remove_hallucinated` / `remove_easy`, then exports MMLU-Pro-style columns to **`benchmark.parquet`**. Review with humans before production use.


In [ ]:
dataset_final = pd.read_parquet(last_path)
dataset_final.to_parquet(output_path_raw)
logger.info("Raw benchmark saved to %s", output_path_raw)

if config.remove_hallucinated:
    n_hallu = dataset_final[dataset_final["is_hallucination"] == True].shape[0]
    dataset_final = dataset_final[dataset_final["is_hallucination"] == False]
    logger.info("Removed %s hallucinated questions.", n_hallu)
if config.remove_easy:
    n_easy = dataset_final[dataset_final["is_easy"] == True].shape[0]
    dataset_final = dataset_final[dataset_final["is_easy"] == False]
    logger.info("Removed %s easy questions.", n_easy)

if len(dataset_final) == 0:
    raise RuntimeError("No questions left after filtering.")

dataset_final = dataset_final[
    [
        "id_question",
        "question_generated",
        "choices_generated",
        "answer_generated",
        "target_subject",
    ]
]
dataset_final = dataset_final.rename(
    columns={
        "id_question": "question_id",
        "question_generated": "question",
        "choices_generated": "options",
        "answer_generated": "answer",
        "target_subject": "category",
    },
)
dataset_final = dataset_final.assign(
    answer_index=dataset_final["answer"].apply(lambda x: ord(x) - ord("A")),
    cot_content="-",
    src="-",
)
dataset_final.to_parquet(output_path_final)

logger.info("Final benchmark saved to %s", output_path_final)
print("\nDone.")
print("  Raw:   ", output_path_raw)
print("  Final: ", output_path_final)
